# 02a: DEVRT Dataset Memory-Efficient Processing

This notebook demonstrates and executes the processing of a single selected DEVRT trip file for schema validation and inspection. By default, it does **not** process the entire dataset automatically.

In [ ]:
import os
import sys
import pandas as pd
import numpy as np
from pathlib import Path

# Ensure project root is in path
project_root = Path(os.getcwd()).parent
sys.path.append(str(project_root))

## Configuration
Specify the sample trip CSV file to process. Modify `SAMPLE_FILE` to inspect any other trip file.

In [ ]:
SAMPLE_FILE = "20230418_DACIA_ANDOAIN_AZPEITIA_011.csv"

## Step 1: Load and Inspect Raw CSV

In [ ]:
vehicle_dir = "DACIA SPRING" if "DACIA" in SAMPLE_FILE else "NISSAN LEAF"
raw_path = project_root / "dataset" / "DEVRT" / "DEVRT" / vehicle_dir / SAMPLE_FILE

raw_df = pd.read_csv(raw_path)
print(f"Raw file: {raw_path.name}")
print(f"Raw file shape: {raw_df.shape[0]} rows, {raw_df.shape[1]} columns\n")
print("Columns present:")
print(list(raw_df.columns))
print("\nFirst 3 rows:")
raw_df.head(3)

## Step 2: Run Parser Standardizations and Quality Checks on Single File

In [ ]:
from src.data.devrt_parser import process_devrt_trip

file_info = {
    'path': str(raw_path),
    'filename': SAMPLE_FILE,
    'trip_name': SAMPLE_FILE.replace('.csv', ''),
    'vehicle': vehicle_dir
}

output_dir = project_root / "data" / "interim" / "devrt"
result = process_devrt_trip(file_info, output_dir=str(output_dir))

print("\n--- Processing Result ---")
for k, v in result.items():
    if k in ['quality_summary', 'missing_summary', 'invalid_summary']:
        print(f"{k}:")
        for subk, subv in v.items():
            if subv > 0:
                print(f"  - {subk}: {subv}")
    else:
        print(f"{k}: {v}")

## Step 3: Load and Inspect Standardized Output Parquet File

In [ ]:
output_parquet = output_dir / f"{SAMPLE_FILE.replace('.csv', '_standardized.parquet')}"
std_df = pd.read_parquet(output_parquet)

print(f"Standardized file: {output_parquet.name}")
print(f"Standardized shape: {std_df.shape[0]} rows, {std_df.shape[1]} columns\n")
print("Standardized schema details:")
print(std_df.info())
print("\nFirst 3 standardized rows:")
std_df.head(3)

## Step 4: Verification of Specific Conversions
Compare some raw vs converted values side-by-side to verify correctness.

In [ ]:
compare_df = pd.DataFrame()
if 'Motor Pwr(w)' in raw_df.columns:
    compare_df['raw_motor_w'] = raw_df['Motor Pwr(w)']
    compare_df['std_motor_kw'] = std_df['motor_power_kw']
if 'Aux Pwr(100w)' in raw_df.columns:
    compare_df['raw_aux_100w'] = raw_df['Aux Pwr(100w)']
    compare_df['std_aux_kw'] = std_df['aux_power_kw']
if 'capacity' in raw_df.columns:
    compare_df['raw_capacity_wh'] = raw_df['capacity']
    compare_df['std_capacity_kwh'] = std_df['battery_capacity_kwh']
if 'regenwh' in raw_df.columns:
    compare_df['raw_regen_w'] = raw_df['regenwh']
    compare_df['std_regen_kw'] = std_df['regen_power_kw']

print("First 5 conversion comparisons (non-zero or non-null samples where possible):")
print(compare_df.dropna(how='all').head(10))